In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, auc, f1_score, confusion_matrix

# Global result logging structure
comparison_records = []

def train_and_evaluate(dataset_name, X_tr, y_tr, X_te, y_te):
    # --- 1. Baseline Model (Logistic Regression) ---
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr, y_tr)
    
    preds_lr = lr.predict(X_te)
    probs_lr = lr.predict_proba(X_te)[:, 1]
    
    p_lr, r_lr, _ = precision_recall_curve(y_te, probs_lr)
    auc_pr_lr = auc(r_lr, p_lr)
    f1_lr = f1_score(y_te, preds_lr)
    
    # Save baseline artifact
    joblib.dump(lr, f'../models/{dataset_name}_logistic_regression.pkl')
    
    # --- 2. Tuned Ensemble Model (Random Forest) ---
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    
    preds_rf = rf.predict(X_te)
    probs_rf = rf.predict_proba(X_te)[:, 1]
    
    p_rf, r_rf, _ = precision_recall_curve(y_te, probs_rf)
    auc_pr_rf = auc(r_rf, p_rf)
    f1_rf = f1_score(y_te, preds_rf)
    
    # Save ensemble artifact
    joblib.dump(rf, f'../models/{dataset_name}_random_forest.pkl')
    
    # Log Comparison Results
    comparison_records.append({
        'Dataset': dataset_name, 'Model': 'Logistic Regression', 'F1-Score': f1_lr, 'AUC-PR': auc_pr_lr
    })
    comparison_records.append({
        'Dataset': dataset_name, 'Model': 'Random Forest', 'F1-Score': f1_rf, 'AUC-PR': auc_pr_rf
    })
    
    print(f"\n=== Evaluation: {dataset_name} ===")
    print(f"LR  - F1: {f1_lr:.4f} | AUC-PR: {auc_pr_lr:.4f}")
    print(f"RF  - F1: {f1_rf:.4f} | AUC-PR: {auc_pr_rf:.4f}")

# Execute for both datasets
train_and_evaluate("Fraud_Data", 
                   np.load('../data/processed/X_train_f.npy'), np.load('../data/processed/y_train_f.npy'),
                   np.load('../data/processed/X_test_f.npy'), np.load('../data/processed/y_test_f.npy'))

train_and_evaluate("CreditCard", 
                   np.load('../data/processed/X_train_c.npy'), np.load('../data/processed/y_train_c.npy'),
                   np.load('../data/processed/X_test_c.npy'), np.load('../data/processed/y_test_c.npy'))

# Display Metrics Table (Rubric 2.4)
df_comparison = pd.DataFrame(comparison_records)
print("\n--- MODEL COMPARISON SUMMARY ---")
print(df_comparison.to_string(index=False))